# Deep Research Agent with Oracle AI Agent Memory (OCI)

[![Documentation](https://img.shields.io/badge/Documentation-Oracle%20AI%20Agent%20Memory-red?style=flat-square)](https://www.oracle.com/database/ai-agent-memory/)

**Framework:** OpenAI Agents SDK over OCI Generative AI · **Web search:** Tavily · **Memory:** Oracle AI Agent Memory on Oracle AI Database

This notebook walks through building a **deep research agent** for human genome exploration. The agent uses OCI Generative AI for generation, Tavily for live web research, and Oracle AI Database via the `oracleagentmemory` package for running conversation state and durable findings.

---

## Application Modes in Agent Applications

AI agent applications generally fall into three operational modes:

| Mode | Shape | Typical use |
|---|---|---|
| **Assistant** | Turn-by-turn conversational | Customer support, coding copilot, chat UIs |
| **Deep Research** | Multi-step autonomous investigation | Literature review, market research, technical scoping |
| **Workflow** | Predetermined sequence of steps, often with conditional branches | Approval pipelines, compliance checks, structured triage |

> **This notebook focuses on the Deep Research mode.**
>
> A deep-research agent plans, retrieves, synthesises, and remembers durable findings across sessions.


## What You'll Learn

- How to run the **OpenAI Agents SDK** against **OCI Generative AI** instead of an hosted model
- How to implement the **OpenAI Agents SDK `Session` protocol** against a custom backend (Oracle AI Agent Memory)
- How to wrap **Tavily** as a `function_tool` the agent can call
- How to store long-lived research findings as durable **memories** (separate from short-term conversation history)
- How to verify memory continuity across sessions — the agent can pick up where it left off

> **💡 Key insight:** Short-term memory (conversation history for the current run) and long-term memory (durable findings across runs) are different access patterns over the same store. We use `Session` for the former and `add_memory()` for the latter.


## Prerequisites

- **Oracle AI Database** running locally (Docker container) or reachable over the network
- **OCI Generative AI** access to `xai.grok-3-fast`
- Default OCI GenAI region is `us-chicago-1`; set `OCI_GENAI_REGION` if your model access lives elsewhere
- Local OCI config file at `~/.oci/config`, or set `OCI_CONFIG_FILE` and `OCI_PROFILE`
- **`OCI_GENAI_COMPARTMENT_OCID`** for native OCI mode, or a `compartment_id`/`tenancy` value in the selected OCI profile
- Optional **`OCI_GENAI_MODEL_OCID`** if you need a specific on-demand model OCID; otherwise the notebook uses `LLM_MODEL` (`xai.grok-3-fast`)
- **`TAVILY_API_KEY`** for web search (free tier available at [tavily.com](https://tavily.com))
- The `oracleagentmemory` wheel installed in the active kernel's environment

This OCI variant does not require a hosted OpenAI API key.


## 1. Install dependencies

Oracle's Agent Memory 26.4 docs install `oracleagentmemory==26.4.0`. We also install the OpenAI Agents SDK, the OCI SDK, Tavily, and notebook async-loop support. The model calls go through the native OCI SDK path below.


In [ ]:
%pip install -q oracleagentmemory==26.4.0 openai-agents openai oci tavily-python nest_asyncio


## 2. Configure OCI, Tavily, and Oracle connection

The notebook defaults to OCI Generative AI native mode and reads credentials from local OCI config. Use environment variables or edit the values below before running the cell. We deliberately do not seed `OCI_CONFIG_FILE`; OCI mode should start without it.


In [ ]:
import configparser
import os
from pathlib import Path


def _expand_path(value: str) -> str:
    return os.path.expanduser(os.path.expandvars(value))


def _read_oci_profile_region(config_file: str, profile_name: str) -> str:
    config_path = Path(config_file).expanduser()
    if not config_path.exists():
        return ""

    parser = configparser.RawConfigParser()
    parser.read(config_path)
    if profile_name == "DEFAULT" and parser.defaults().get("region"):
        return parser.defaults()["region"].strip()
    for section in (profile_name, f"profile {profile_name}"):
        if parser.has_section(section) and parser.has_option(section, "region"):
            return parser.get(section, "region").strip()
    return ""


def _read_oci_profile_values(config_file: str, profile_name: str) -> dict[str, str]:
    config_path = Path(config_file).expanduser()
    if not config_path.exists():
        return {}

    parser = configparser.RawConfigParser()
    parser.read(config_path)
    if profile_name == "DEFAULT":
        values = dict(parser.defaults())
    else:
        values = {}
        for section in (profile_name, f"profile {profile_name}"):
            if parser.has_section(section):
                values.update(dict(parser.items(section)))
                break

    if values.get("key_file"):
        values["key_file"] = _expand_path(values["key_file"])
    return values


def _set_oci_env_aliases(config_file: str, profile_name: str) -> None:
    values = _read_oci_profile_values(config_file, profile_name)
    aliases = {
        "user": ("OCI_USER", "OCI_USER_ID"),
        "tenancy": ("OCI_TENANCY", "OCI_TENANCY_ID"),
        "fingerprint": ("OCI_FINGERPRINT",),
        "key_file": ("OCI_KEY_FILE",),
        "region": ("OCI_REGION", "OCI_REGION_NAME"),
    }
    for config_key, env_names in aliases.items():
        value = values.get(config_key, "")
        if not value:
            continue
        for env_name in env_names:
            os.environ.setdefault(env_name, value)


def _env(name: str, default: str = "") -> str:
    raw_value = os.environ.get(name)
    value = raw_value if raw_value not in (None, "") else default
    value = _expand_path(value) if name.endswith("_FILE") else value
    os.environ[name] = value
    return value


def _require_non_empty(value: str, name: str, hint: str) -> None:
    if not value:
        raise RuntimeError(f"Missing {name}. {hint}")


def _require_existing_file(value: str, name: str, hint: str) -> None:
    _require_non_empty(value, name, hint)
    if not Path(value).expanduser().exists():
        raise RuntimeError(f"{name} does not exist at {value}. {hint}")


LLM_PROVIDER = _env("LLM_PROVIDER", "oci_genai_native").strip().lower().replace("-", "_")
if LLM_PROVIDER not in {"oci_genai_native", "oci_native"}:
    raise RuntimeError("This OCI notebook supports LLM_PROVIDER=oci_genai_native only.")

LLM_MODEL = _env("LLM_MODEL", "xai.grok-3-fast").strip()
OCI_CONFIG_FILE = _env("OCI_CONFIG_FILE", "~/.oci/config")
OCI_PROFILE = _env("OCI_PROFILE", "DEFAULT")
OCI_PROFILE_REGION = _read_oci_profile_region(OCI_CONFIG_FILE, OCI_PROFILE)
OCI_GENAI_REGION = os.environ.get("OCI_GENAI_REGION", "").strip() or "us-chicago-1"
os.environ["OCI_GENAI_REGION"] = OCI_GENAI_REGION
OCI_GENAI_ENDPOINT = _env(
    "OCI_GENAI_ENDPOINT",
    f"https://inference.generativeai.{OCI_GENAI_REGION}.oci.oraclecloud.com",
)
OCI_PROFILE_VALUES = _read_oci_profile_values(OCI_CONFIG_FILE, OCI_PROFILE)
OCI_GENAI_COMPARTMENT_OCID = _env(
    "OCI_GENAI_COMPARTMENT_OCID",
    OCI_PROFILE_VALUES.get("compartment_id") or OCI_PROFILE_VALUES.get("tenancy", ""),
).strip()
OCI_GENAI_MODEL_OCID = _env("OCI_GENAI_MODEL_OCID", LLM_MODEL).strip()
OCI_GENAI_EMBED_MODEL = _env("OCI_GENAI_EMBED_MODEL", "cohere.embed-english-v3.0")

_set_oci_env_aliases(OCI_CONFIG_FILE, OCI_PROFILE)
os.environ.setdefault("OCI_REGION", OCI_GENAI_REGION)
os.environ.setdefault("OCI_REGION_NAME", OCI_GENAI_REGION)
os.environ.setdefault("OCI_COMPARTMENT_ID", OCI_GENAI_COMPARTMENT_OCID)

os.environ.setdefault("DB_USER", "VECTOR")
os.environ.setdefault("DB_PASSWORD", "VectorPwd_2025")
os.environ.setdefault("DB_CONNECT_STRING", "localhost:1521/FREEPDB1")

_require_existing_file(
    OCI_CONFIG_FILE,
    "OCI_CONFIG_FILE",
    "Set OCI_CONFIG_FILE to your OCI config path, usually ~/.oci/config.",
)
_require_non_empty(OCI_PROFILE, "OCI_PROFILE", "Set OCI_PROFILE to a profile in OCI_CONFIG_FILE.")
_require_non_empty(OCI_GENAI_REGION, "OCI_GENAI_REGION", "Set OCI_GENAI_REGION for OCI Generative AI.")
_require_non_empty(
    OCI_GENAI_COMPARTMENT_OCID,
    "OCI_GENAI_COMPARTMENT_OCID",
    "Set this to a compartment OCID with OCI Generative AI permissions. The notebook falls back to compartment_id or tenancy from OCI_CONFIG_FILE when present.",
)
_require_non_empty(
    OCI_GENAI_MODEL_OCID,
    "OCI_GENAI_MODEL_OCID",
    "Set this to the on-demand model OCID or model ID. Defaults to LLM_MODEL, for example xai.grok-3-fast.",
)

import nest_asyncio
nest_asyncio.apply()

print("Environment configured for OCI Generative AI.")
print(f"LLM provider: {LLM_PROVIDER}")
print(f"LLM model: {LLM_MODEL}")
print(f"OCI profile region: {OCI_PROFILE_REGION or 'not set'}")
print(f"OCI GenAI region: {OCI_GENAI_REGION}")
print(f"OCI endpoint: {OCI_GENAI_ENDPOINT}")
print(f"OCI compartment: {OCI_GENAI_COMPARTMENT_OCID[:18]}..." if OCI_GENAI_COMPARTMENT_OCID else "OCI compartment: not set")
print(f"OCI model id: {OCI_GENAI_MODEL_OCID}")
print(f"OCI embedding model: {OCI_GENAI_EMBED_MODEL}")


TAVILY_API_KEY = os.environ.get("TAVILY_API_KEY", "").strip()
if not TAVILY_API_KEY or TAVILY_API_KEY == "tvly-":
    import getpass

    TAVILY_API_KEY = getpass.getpass("TAVILY_API_KEY: ").strip()
    if TAVILY_API_KEY:
        os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY
print("Tavily API key: configured" if os.environ.get("TAVILY_API_KEY") else "Tavily API key: not set")


## 3. Connect to Oracle AI Database and initialise the memory client

`OracleAgentMemory` is the governed memory client. This OCI version uses a small native OCI SDK embedder for semantic search and disables automatic extractor LLM calls; durable findings are written explicitly through the `save_research_finding` tool later in the notebook. That keeps generation and embeddings on OCI without a hidden `OCI_CONFIG_FILE` or LiteLLM dependency.

> **💡 Key insight:** `schema_policy` controls how the library interacts with the DB on startup. `REQUIRE_EXISTING` (default) never issues DDL — safe for production. `CREATE_IF_NECESSARY` fills in missing objects — safe for dev. `RECREATE` drops and rebuilds — useful when you've changed embedding dimensions.


In [ ]:
import asyncio

import numpy as np
import oci
import oracledb
from oracleagentmemory.apis.embedders.embedder import IEmbedder
from oracleagentmemory.core import OracleAgentMemory


class OciSdkEmbedder(IEmbedder):
    """OracleAgentMemory embedder backed directly by OCI Generative AI.

    The Oracle docs show oracleagentmemory.core.embedders.Embedder for generic
    provider adapters. This notebook avoids that adapter and calls the OCI SDK
    directly so there is no LiteLLM embedding configuration to maintain.
    """

    def __init__(self, client, compartment_id: str, model_id: str):
        self.client = client
        self.compartment_id = compartment_id
        self.model_id = model_id

    def embed(self, texts: list[str], *, is_query: bool = False) -> np.ndarray:
        from oci.generative_ai_inference import models

        details = models.EmbedTextDetails()
        details.inputs = texts
        details.compartment_id = self.compartment_id
        details.serving_mode = models.OnDemandServingMode(model_id=self.model_id)
        details.truncate = models.EmbedTextDetails.TRUNCATE_END
        details.input_type = (
            models.EmbedTextDetails.INPUT_TYPE_SEARCH_QUERY
            if is_query
            else models.EmbedTextDetails.INPUT_TYPE_SEARCH_DOCUMENT
        )

        response = self.client.embed_text(details)
        embeddings = getattr(response.data, "embeddings", None)
        if not embeddings:
            raise RuntimeError("OCI embed_text returned no embeddings.")
        return np.asarray(embeddings, dtype=np.float32)

    async def embed_async(self, texts: list[str], *, is_query: bool = False) -> np.ndarray:
        return await asyncio.to_thread(self.embed, texts, is_query=is_query)


connection = oracledb.connect(
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    dsn=os.environ["DB_CONNECT_STRING"],
)
print("Connected to Oracle AI Database:", connection.version)

oci_config = oci.config.from_file(OCI_CONFIG_FILE, OCI_PROFILE)
genai_client = oci.generative_ai_inference.GenerativeAiInferenceClient(
    config=oci_config,
    service_endpoint=OCI_GENAI_ENDPOINT,
    retry_strategy=oci.retry.NoneRetryStrategy(),
    timeout=(10, 240),
)

oamp_embedder = OciSdkEmbedder(
    genai_client,
    compartment_id=OCI_GENAI_COMPARTMENT_OCID,
    model_id=OCI_GENAI_EMBED_MODEL,
)

memory_client = OracleAgentMemory(
    connection=connection,
    embedder=oamp_embedder,
    extract_memories=False,              # durable findings are saved by tool calls
    schema_policy="recreate",
    table_name_prefix="genome_",        # isolates this example's tables from others
)
print("Memory client ready.")


## 4. Register the research user and agent

Oracle AI Agent Memory scopes every record by `user_id` and `agent_id`. Registering them up-front gives the store a stable identity to hang memories on and lets us enforce tenant-style isolation across multiple users.

> **📌 Scoping matters for production.** In a multi-user deployment, `user_id` should match your application's authenticated user. The memory client's `search()` method requires a concrete `user_id` on every call — an intentional guardrail against cross-tenant leaks.

In [ ]:
USER_ID = "genome-researcher-1"
AGENT_ID = "deep-research-agent"

for create_fn, eid, info in [
    (memory_client.add_user, USER_ID, "Genomics researcher exploring human disease associations."),
    (memory_client.add_agent, AGENT_ID, "Deep-research agent with web search and long-term memory."),
]:
    try:
        create_fn(eid, info)
        print(f"Registered: {eid}")
    except ValueError:
        print(f"Already exists: {eid}")

## 5. Implement the `Session` protocol on top of Oracle AI Agent Memory

The OpenAI Agents SDK defines a `Session` protocol with four async methods:

| Method | Purpose |
|---|---|
| `get_items(limit)` | Return the conversation history the runner should inject |
| `add_items(items)` | Persist new items the runner produced during a run |
| `pop_item()` | Remove and return the most recent item (for corrections) |
| `clear_session()` | Drop everything for this session |

Implementing this protocol against Oracle AI Agent Memory gives us **exact-resumption** short-term memory — the next time this session runs, the agent receives the full prior conversation without any manual wiring.

> **💡 Key insight:** The `Session` protocol handles *short-term* memory (the items the runner replays each turn). Long-term memory (durable facts the agent should remember across sessions) is a separate concern that we handle with `add_memory()` below.

In [ ]:
import json
from typing import Optional
from agents.memory.session import SessionABC


class OracleAgentMemorySession(SessionABC):
    """Session backend that persists OpenAI Agents SDK items in Oracle AI Agent Memory.

    Each item is serialised to JSON and stored as a memory record tagged with the
    session's thread_id. Ordering is preserved by the store's insertion timestamp.
    """

    def __init__(self, session_id: str, client: OracleAgentMemory, user_id: str, agent_id: str):
        self.session_id = session_id
        self._client = client
        self._store = client._store
        self._user_id = user_id
        self._agent_id = agent_id
        # Ensure the underlying thread exists so memories can attach to it
        try:
            self._client.create_thread(
                thread_id=session_id, user_id=user_id, agent_id=agent_id,
            )
        except ValueError:
            pass  # thread already exists

    async def get_items(self, limit: Optional[int] = None) -> list:
        records = self._store.list(
            "memory",
            thread_id=self.session_id,
            user_id=self._user_id,
            agent_id=self._agent_id,
            limit=limit if limit else 10_000,
            metadata_filter={"session_item": True},
        )
        items = [json.loads(r.content) for r in records]
        return items[-limit:] if limit else items

    async def add_items(self, items: list) -> None:
        if not items:
            return
        for item in items:
            self._client.add_memory(
                json.dumps(item),
                user_id=self._user_id,
                agent_id=self._agent_id,
                thread_id=self.session_id,
                metadata={"session_item": True},
            )

    async def pop_item(self) -> Optional[dict]:
        records = self._store.list(
            "memory",
            thread_id=self.session_id,
            user_id=self._user_id,
            agent_id=self._agent_id,
            limit=1,
            metadata_filter={"session_item": True},
        )
        if not records:
            return None
        # Records come back newest-first when limit=1 is applied; delete and return
        last = records[-1]
        self._store.delete("memory", last.id)
        return json.loads(last.content)

    async def clear_session(self) -> None:
        records = self._store.list(
            "memory",
            thread_id=self.session_id,
            user_id=self._user_id,
            agent_id=self._agent_id,
            limit=10_000,
            metadata_filter={"session_item": True},
        )
        for r in records:
            self._store.delete("memory", r.id)


print("OracleAgentMemorySession implemented.")

## 6. Define the agent's tools

The agent needs three tools:

1. **`tavily_search`** — live web search for up-to-date genomics sources
2. **`save_research_finding`** — persist a durable research note the agent can recall in future sessions
3. **`recall_research_findings`** — search the long-term memory for prior findings on a topic

Tools 2 and 3 are how the agent manages its **long-term** memory. They're distinct from the `Session` machinery, which handles short-term conversation history automatically.

> **📌 The `@function_tool` decorator** auto-generates the tool's JSON schema from Python type hints and parses the docstring for the description the LLM sees. Write informative docstrings — the LLM reads them.

In [ ]:
from agents import function_tool
from tavily import TavilyClient

TAVILY_API_KEY = os.environ.get("TAVILY_API_KEY", "").strip()
if not TAVILY_API_KEY or TAVILY_API_KEY == "tvly-":
    raise RuntimeError(
        "Missing TAVILY_API_KEY. Set it in the notebook environment before defining tools; "
        "otherwise the research agent cannot fetch current sources."
    )

_tavily = TavilyClient(api_key=TAVILY_API_KEY)

@function_tool
def tavily_search(query: str, max_results: int = 5) -> str:
    """Search the live web for recent, authoritative genomics sources.

    Use this to pull current information from NCBI, OMIM, peer-reviewed journals,
    and other scientific sources. Prefer this over relying on model-internal knowledge
    for facts about specific genes, mutations, or recent studies.

    Args:
        query: Natural-language search query.
        max_results: How many results to return (1-10).
    """
    try:
        resp = _tavily.search(
            query=query, max_results=max_results,
            search_depth="advanced", include_answer=True,
        )
    except Exception as exc:
        return f"Tavily search failed: {type(exc).__name__}: {exc}"

    lines = [f"Answer: {resp.get('answer', '')}"]
    for r in resp.get("results", []):
        lines.append(f"- {r['title']} ({r['url']})\n  {r['content'][:300]}")
    return "\n".join(lines)


@function_tool
def save_research_finding(topic: str, finding: str) -> str:
    """Persist a durable research finding the agent should recall in future sessions.

    Use this for claims worth remembering across sessions — e.g. "BRCA1 mutations
    are associated with 45-85% lifetime breast cancer risk in women." Do not use
    this for conversational acknowledgements or speculation.

    Args:
        topic: Short topic tag (e.g. 'BRCA1', 'TP53 mutations').
        finding: The finding to remember, ideally one sentence.
    """
    memory_id = memory_client.add_memory(
        f"[{topic}] {finding}",
        user_id=USER_ID,
        agent_id=AGENT_ID,
        metadata={"topic": topic, "kind": "research_finding"},
    )
    return f"Saved finding (id={memory_id})."


@function_tool
def recall_research_findings(query: str, max_results: int = 5) -> str:
    """Search prior research findings for information relevant to a query.

    Use this at the start of a research task to check what is already known.
    Results are ranked by semantic similarity.

    Args:
        query: Natural-language query describing what you want to recall.
        max_results: How many findings to return.
    """
    results = memory_client.search(
        query, user_id=USER_ID, agent_id=AGENT_ID, max_results=max_results,
    )
    if not results:
        return "(no prior findings)"
    return "\n".join(
        f"- {r.content}  [distance={r.distance:.3f}]" for r in results
    )


print("Tools defined: tavily_search, save_research_finding, recall_research_findings")

## 7. Construct the OCI-backed research agent

The OpenAI Agents SDK expects a Chat Completions-shaped model client. The adapter below keeps that interface but sends every generation request to OCI Generative AI through the native OCI Python SDK.


In [ ]:
import asyncio
import json
import time
import uuid
from types import SimpleNamespace
from typing import Any

from agents import OpenAIChatCompletionsModel, set_tracing_disabled
from openai.types.chat import ChatCompletion, ChatCompletionChunk

set_tracing_disabled(True)


def _param_or_default(value: Any, default: Any) -> Any:
    if value is None:
        return default
    if value.__class__.__name__ in {"Omit", "NotGiven"}:
        return default
    return value


def _openai_tool_call_to_oci(models, tool_call: dict):
    function = tool_call.get("function", {}) if isinstance(tool_call, dict) else {}
    call = models.FunctionCall()
    if hasattr(models.FunctionCall, "TYPE_FUNCTION"):
        call.type = models.FunctionCall.TYPE_FUNCTION
    call.id = tool_call.get("id", "") if isinstance(tool_call, dict) else ""
    call.name = function.get("name", "")
    arguments = function.get("arguments", "{}")
    call.arguments = json.dumps(arguments) if isinstance(arguments, dict) else str(arguments or "{}")
    return call


def _openai_message_to_oci(models, message: dict):
    role = (message.get("role") or "user").lower()
    content_items = [_oci_text_content(models, message.get("content"))] if message.get("content") is not None else []

    if role == "system":
        oci_message = models.SystemMessage()
        oci_message.role = models.SystemMessage.ROLE_SYSTEM
    elif role == "assistant":
        oci_message = models.AssistantMessage()
        oci_message.role = models.AssistantMessage.ROLE_ASSISTANT
        if message.get("tool_calls"):
            oci_message.tool_calls = [
                _openai_tool_call_to_oci(models, tool_call)
                for tool_call in message.get("tool_calls", [])
            ]
    elif role == "tool":
        oci_message = models.ToolMessage()
        oci_message.role = models.ToolMessage.ROLE_TOOL
        oci_message.tool_call_id = message.get("tool_call_id", "")
    else:
        oci_message = models.UserMessage()
        oci_message.role = models.UserMessage.ROLE_USER

    oci_message.content = content_items
    return oci_message


def _openai_tool_to_oci(models, tool: dict):
    function = tool.get("function", {}) if isinstance(tool, dict) else {}
    definition = models.FunctionDefinition()
    if hasattr(models.FunctionDefinition, "TYPE_FUNCTION"):
        definition.type = models.FunctionDefinition.TYPE_FUNCTION
    definition.name = function.get("name", "")
    definition.description = function.get("description", "")
    definition.parameters = function.get("parameters", {})
    return definition


def _openai_tool_choice_to_oci(models, tool_choice: Any):
    if not tool_choice or tool_choice == "auto":
        choice = models.ToolChoiceAuto()
        if hasattr(models.ToolChoiceAuto, "TYPE_AUTO"):
            choice.type = models.ToolChoiceAuto.TYPE_AUTO
        return choice
    if tool_choice == "none":
        choice = models.ToolChoiceNone()
        if hasattr(models.ToolChoiceNone, "TYPE_NONE"):
            choice.type = models.ToolChoiceNone.TYPE_NONE
        return choice
    if tool_choice == "required":
        choice = models.ToolChoiceRequired()
        if hasattr(models.ToolChoiceRequired, "TYPE_REQUIRED"):
            choice.type = models.ToolChoiceRequired.TYPE_REQUIRED
        return choice
    if isinstance(tool_choice, dict):
        function = tool_choice.get("function", {})
        if function.get("name"):
            choice = models.ToolChoiceFunction()
            if hasattr(models.ToolChoiceFunction, "TYPE_FUNCTION"):
                choice.type = models.ToolChoiceFunction.TYPE_FUNCTION
            choice.name = function["name"]
            return choice
    return None


def _extract_oci_tool_calls(message: Any) -> list[dict]:
    converted = []
    for index, call in enumerate(getattr(message, "tool_calls", None) or []):
        arguments = getattr(call, "arguments", "{}") or "{}"
        if not isinstance(arguments, str):
            arguments = json.dumps(arguments)
        converted.append({
            "id": getattr(call, "id", "") or f"oci-tool-call-{index}",
            "type": "function",
            "function": {
                "name": getattr(call, "name", "") or "",
                "arguments": arguments,
            },
        })
    return converted


def _chat_completion_from_oci(model: str, message: Any) -> ChatCompletion:
    content = _extract_oci_text(message)
    tool_calls = _extract_oci_tool_calls(message)
    assistant_message = {
        "role": "assistant",
        "content": None if tool_calls else content,
    }
    if tool_calls:
        assistant_message["tool_calls"] = tool_calls

    return ChatCompletion(
        id=f"oci-{uuid.uuid4()}",
        object="chat.completion",
        created=int(time.time()),
        model=model,
        choices=[{
            "index": 0,
            "finish_reason": "tool_calls" if tool_calls else "stop",
            "message": assistant_message,
        }],
    )


async def _chat_completion_as_stream(completion: ChatCompletion):
    choice = completion.choices[0]
    message = choice.message
    chunk_id = f"{completion.id}-chunk"
    created = int(time.time())

    if message.tool_calls:
        delta_tool_calls = []
        for index, call in enumerate(message.tool_calls):
            delta_tool_calls.append({
                "index": index,
                "id": call.id,
                "type": "function",
                "function": {
                    "name": call.function.name,
                    "arguments": call.function.arguments,
                },
            })
        yield ChatCompletionChunk(
            id=chunk_id,
            object="chat.completion.chunk",
            created=created,
            model=completion.model,
            choices=[{
                "index": 0,
                "delta": {"role": "assistant", "tool_calls": delta_tool_calls},
                "finish_reason": None,
            }],
        )
        finish_reason = "tool_calls"
    else:
        yield ChatCompletionChunk(
            id=chunk_id,
            object="chat.completion.chunk",
            created=created,
            model=completion.model,
            choices=[{
                "index": 0,
                "delta": {"role": "assistant", "content": message.content or ""},
                "finish_reason": None,
            }],
        )
        finish_reason = "stop"

    yield ChatCompletionChunk(
        id=chunk_id,
        object="chat.completion.chunk",
        created=created,
        model=completion.model,
        choices=[{
            "index": 0,
            "delta": {},
            "finish_reason": finish_reason,
        }],
    )


class OciNativeChatCompletions:
    """Async Chat Completions-shaped adapter backed by OCI GenAI native Chat API."""

    def __init__(self, client, compartment_id: str, model_id: str, default_model: str):
        self.client = client
        self.compartment_id = compartment_id
        self.model_id = model_id
        self.default_model = default_model

    async def create(
        self,
        *,
        model: str | None = None,
        messages: list[dict] | None = None,
        tools: list[dict] | None = None,
        tool_choice: Any = None,
        stream: bool = False,
        **kwargs,
    ):
        completion = await asyncio.to_thread(
            self._create_sync,
            model or self.default_model,
            messages or [],
            tools or [],
            tool_choice,
            kwargs,
        )
        if stream:
            return _chat_completion_as_stream(completion)
        return completion

    def _create_sync(self, model: str, messages: list[dict], tools: list[dict], tool_choice: Any, kwargs: dict):
        from oci.generative_ai_inference import models

        chat_request = models.GenericChatRequest()
        chat_request.api_format = models.BaseChatRequest.API_FORMAT_GENERIC
        chat_request.messages = [_openai_message_to_oci(models, message) for message in messages]
        chat_request.max_tokens = _param_or_default(
            kwargs.get("max_tokens"),
            _param_or_default(kwargs.get("max_completion_tokens"), 1200),
        )
        chat_request.temperature = _param_or_default(kwargs.get("temperature"), 0.2)
        chat_request.top_p = _param_or_default(kwargs.get("top_p"), 1.0)
        chat_request.top_k = _param_or_default(kwargs.get("top_k"), 0)
        chat_request.is_stream = False

        if tools:
            chat_request.tools = [_openai_tool_to_oci(models, tool) for tool in tools]
            native_tool_choice = _openai_tool_choice_to_oci(models, tool_choice)
            if native_tool_choice:
                chat_request.tool_choice = native_tool_choice

        serving_mode = models.OnDemandServingMode()
        serving_mode.model_id = self.model_id

        chat_detail = models.ChatDetails()
        chat_detail.serving_mode = serving_mode
        chat_detail.chat_request = chat_request
        chat_detail.compartment_id = self.compartment_id

        response = self.client.chat(chat_detail)
        data = response.data
        chat_response = getattr(data, "chat_response", data)
        choices = getattr(chat_response, "choices", None) or []
        message = getattr(choices[0], "message", None) if choices else None
        return _chat_completion_from_oci(model, message)


class OciNativeAsyncOpenAI:
    """Enough of AsyncOpenAI's shape for OpenAIChatCompletionsModel."""

    def __init__(self, native_client, endpoint: str, compartment_id: str, model_id: str, default_model: str):
        self.base_url = endpoint
        self.chat = SimpleNamespace(
            completions=OciNativeChatCompletions(
                native_client,
                compartment_id=compartment_id,
                model_id=model_id,
                default_model=default_model,
            )
        )

    def with_options(self, **_kwargs):
        return self


def create_agent_model():
    openai_shaped_client = OciNativeAsyncOpenAI(
        genai_client,
        endpoint=OCI_GENAI_ENDPOINT,
        compartment_id=OCI_GENAI_COMPARTMENT_OCID,
        model_id=OCI_GENAI_MODEL_OCID,
        default_model=LLM_MODEL,
    )
    return OpenAIChatCompletionsModel(model=LLM_MODEL, openai_client=openai_shaped_client)


from agents import Agent, Runner

INSTRUCTIONS = """You are a deep-research agent specialising in human genome exploration.

For every research question:
1. FIRST call `recall_research_findings` to check what is already known from prior sessions.
2. If the prior findings are insufficient or outdated, call `tavily_search` for current sources.
3. Synthesise a clear, cited answer. Prefer authoritative sources (NCBI, OMIM, PubMed).
4. Call `save_research_finding` for each durable conclusion worth remembering across sessions.
   Save one finding per call; keep findings atomic and one sentence long.
5. Present the final answer to the user with inline citations (URLs).

Do not save conversational acknowledgements as findings. Only save factual conclusions.
"""

agent_model = create_agent_model()
research_agent = Agent(
    name="GenomeDeepResearcher",
    instructions=INSTRUCTIONS,
    tools=[tavily_search, save_research_finding, recall_research_findings],
    model=agent_model,
)
print(f"Agent created: {research_agent.name} using {LLM_PROVIDER}/{LLM_MODEL}")


## 8. Run a research session

We create a session (tied to a `thread_id` in the memory store) and run the agent over a sequence of research questions. Because we pass `session=session`, the SDK will automatically inject prior conversation items at the start of each turn and persist new items at the end.

> **📌 Separation of concerns.**
> - The `Session` persists the **raw conversation** (working memory).
> - The agent's `save_research_finding` tool persists **durable findings** (long-term memory).
> Both live in the same Oracle database but are accessed through different interfaces.

In [ ]:
SESSION_ID = "genome-session-001"
session = OracleAgentMemorySession(
    session_id=SESSION_ID, 
    client=memory_client,
    user_id=USER_ID, 
    agent_id=AGENT_ID,
)

research_questions = [
    "What is BRCA1 and what is its primary function in DNA repair?",
    "What is the typical lifetime breast cancer risk associated with pathogenic BRCA1 variants?",
    "How does BRCA1 interact with BRCA2 in homologous recombination?",
]

for i, q in enumerate(research_questions, 1):
    print(f"\n{'=' * 70}\nQ{i}: {q}\n{'=' * 70}")
    result = await Runner.run(research_agent, q, session=session)
    print(result.final_output)

## 9. Inspect what the agent remembered

At this point the agent has accumulated both short-term conversation items (via the session) and long-term research findings (via `save_research_finding`). Let's inspect both.

In [ ]:
# Short-term: conversation items replayed each turn
session_items = await session.get_items()
print(f"Session items: {len(session_items)}")
for it in session_items[:3]:
    print(f"  - {str(it)[:120]}...")

# Long-term: durable findings the agent chose to save
findings = memory_client.search(
    "BRCA1 function and risk", user_id=USER_ID, agent_id=AGENT_ID, max_results=10,
)
print(f"\nDurable research findings: {len(findings)}")
for r in findings:
    print(f"  - {r.content}  [distance={r.distance:.3f}]")

## 10. Verify continuity — resume a fresh session with the same memory store

The real test of a memory-aware agent is whether it can pick up where a prior session left off. Let's create a *new* session (simulating a separate process or later day) and ask a question that builds on prior findings.

If the agent recalls BRCA1 findings from the previous run without re-searching, we've demonstrated end-to-end memory continuity.

In [ ]:
# Simulate a fresh session (new session_id, but same user/agent)
FRESH_SESSION_ID = "genome-session-002"
fresh_session = OracleAgentMemorySession(
    session_id=FRESH_SESSION_ID, client=memory_client,
    user_id=USER_ID, agent_id=AGENT_ID,
)

followup = "Based on what you already know about BRCA1, explain how a patient with a pathogenic BRCA1 variant might be counselled about screening."

print(f"FRESH SESSION — Q: {followup}\n")
result = await Runner.run(research_agent, followup, session=fresh_session)
print(result.final_output)

## 11. Cleanup (optional)

Uncomment the cell below to clear all session items and findings for this example. Leave it commented to keep the data for subsequent runs — that's often the point.

In [ ]:
# await session.clear_session()
# await fresh_session.clear_session()
# for r in memory_client.search("BRCA", user_id=USER_ID, agent_id=AGENT_ID, max_results=100):
#     memory_client.delete_memory(r.record.id)
# print("Cleaned up.")

connection.close()
print("Connection closed.")

## Key Takeaways

> **🎯 1. The `Session` protocol is the clean integration point.** Four async methods (`get_items`, `add_items`, `pop_item`, `clear_session`) is all the OpenAI Agents SDK needs to plug a custom memory backend in. You don't have to modify the runner — you implement the protocol and pass an instance via `session=`.

> **🎯 2. Separate short-term and long-term memory concerns.** The session handles conversation replay for the current agent loop. Durable findings (the ones worth recalling in a future session) should be written through tools the agent calls deliberately, not dumped into the conversation log.

> **🎯 3. Instructions steer memory usage.** A deep-research agent must be explicitly told to *recall before researching* and *save durable conclusions*. Otherwise the LLM will treat memory as optional decoration and the store will fill up with nothing useful.

> **🎯 4. Oracle AI Database gives you one substrate for both tiers.** Short-term items, long-term findings, and any structured business data the agent needs to reference all live in one governed backend — one pool, one backup, one compliance review.

> **🎯 5. Continuity is testable.** A new `session_id` with the same `user_id` / `agent_id` should produce an agent that reasons over prior long-term findings. That's the simplest end-to-end test of a memory-aware agent.